$\newcommand{\ket}[1]{\left\lvert #1\right\rangle}
\newcommand{\bra}[1]{\left\langle #1\right\rvert}
\newcommand{\braket}[1]{\left\langle #1\right\rangle}
\newcommand{\proj}[1]{\lvert #1\rangle\!\langle #1\rvert}
\newcommand{\tr}{\operatorname{tr}}\newcommand{\Tr}{\operatorname{Tr}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\rank}{\operatorname{rank}}\newcommand{\diag}{\operatorname{diag}}
\newcommand{\poly}{\operatorname{poly}}\newcommand{\polylog}{\operatorname{polylog}}
\newcommand{\norm}[1]{\lVert #1\rVert}\newcommand{\abs}[1]{\lvert #1\rvert}
\newcommand{\bigo}{\mathcal{O}}\newcommand{\id}{\mathbb{I}}\newcommand{\dg}{^{\dagger}}
\newcommand{\R}{\mathbb{R}}\newcommand{\C}{\mathbb{C}}\newcommand{\N}{\mathbb{N}}
\newcommand{\Z}{\mathbb{Z}}\newcommand{\E}{\mathbb{E}}\newcommand{\Prob}{\mathbb{P}}
\newcommand{\Hilb}{\mathcal{H}}\newcommand{\ii}{\mathrm{i}}\newcommand{\dd}{\mathrm{d}}
\newcommand{\ee}{\mathrm{e}}\newcommand{\QFT}{\mathrm{QFT}}\newcommand{\CNOT}{\mathrm{CNOT}}
\newcommand{\Uf}{U_f}\newcommand{\defeq}{:=}
\newcommand{\Xgate}{\mathsf{X}}\newcommand{\Ygate}{\mathsf{Y}}
\newcommand{\Zgate}{\mathsf{Z}}\newcommand{\Hgate}{\mathsf{H}}$

# Open Quantum Systems and Nonlinear PDEs

**Learning objectives.**

-   Describe Lindblad dynamics and its dilation to a unitary.

-   Prepare thermal and ground states dissipatively.

-   Outline how nonlinear PDEs are linearized (Carleman, Cole--Hopf) for quantum solvers.

The last three chapters kept the dynamics *linear*: the Schrödinger equation
(that chapter) and the general linear ODE $\dot x=Ax+b$ (that chapter). This
chapter relaxes both idealizations, in two independent directions that nonetheless share a common
strategy---*embed the hard problem inside a larger linear or unitary one*. The first half
treats *open* quantum systems, whose state is a density operator $\rho$ evolving under a
non-unitary but completely positive semigroup; the governing equation is the Lindblad master
equation, the continuous-time analogue of the CPTP channels of that chapter. We simulate it
by dilating its non-unitary generator to a unitary on a larger space (Stinespring,
that theorem) and by operator splitting and a truncated Dyson series, all
reusing the Hamiltonian simulation of that chapter. We then turn dissipation into a
resource: by engineering the jump operators so that a target thermal or ground state is the
*unique fixed point*, Lindblad dynamics prepares that state, an open-system cousin of the
adiabatic and Zeno solvers of that chapter. The second half attacks *nonlinear* PDEs.
Because quantum mechanics is linear, a nonlinear evolution must first be *linearized*: we
present Carleman linearization, which embeds a quadratic ODE into an infinite linear system on
the tensor powers of the state and truncates it (solving the result with the methods of
that chapter), and the exact Cole--Hopf transformation, which maps the viscous Burgers
equation onto the heat equation. Throughout, $\norm\cdot$ is the spectral norm, $\norm\cdot_1$
the trace norm, $\norm\cdot_\diamond$ the diamond norm of that chapter, and $\mathcal D(\Hilb)$
the set of density operators on $\Hilb$.

## Lindblad dynamics

##### The master equation.

An open system interacts with environmental degrees of freedom that are not modeled explicitly.
In the Markovian regime---environmental correlations decay fast on the system's timescale---the
state $\rho(t)\in\mathcal D(\C^d)$ obeys the Gorini--Kossakowski--Sudarshan--Lindblad (GKSL)
master equation
$$\frac{\dd\rho}{\dd t}
  =\underbrace{-\ii[H,\rho]}_{\mathcal L_H(\rho)}
   +\underbrace{\sum_{j=1}^{J}\Bigl(L_j\rho L_j\dg-\tfrac12\bigl\{L_j\dg L_j,\rho\bigr\}\Bigr)}
     _{\mathcal L_D(\rho)}
  =:\mathcal L(\rho),$$
where $H=H\dg$ is the system Hamiltonian, the $L_j\in\C^{d\times d}$ are *jump operators*
encoding the coupling to the environment, $[A,B]=AB-BA$, and $\{A,B\}=AB+BA$. The generator
$\mathcal L$ is the *Lindbladian*; its coherent part $\mathcal L_H$ reproduces the
Liouville--von Neumann equation, while its dissipative part $\mathcal L_D$ is genuinely
irreversible. The formal solution is the one-parameter semigroup
$$\rho(t)=\Phi_t\bigl(\rho(0)\bigr),\qquad \Phi_t:=\ee^{\mathcal L t},\quad t\ge0.$$
When all $L_j=0$ this collapses to $\rho(t)=\ee^{-\ii Ht}\rho(0)\ee^{\ii Ht}$, closed-system
unitary evolution. The value of the Lindblad form is that $\Phi_t$ is always a legitimate
channel.

**Theorem ($\ee^{\mathcal L t}$ is a quantum channel).**

For every $t\ge0$ the map $\Phi_t=\ee^{\mathcal L t}:L(\C^d)\to L(\C^d)$ defined by
the equation above--the equation above is completely positive and trace preserving
(CPTP), i.e. a quantum channel in the sense of that definition. The family
$\{\Phi_t\}_{t\ge0}$ forms a semigroup: $\Phi_0=\mathrm{id}$ and $\Phi_{t+s}=\Phi_t\Phi_s$.

*Proof.*

The semigroup laws are immediate from $\ee^{\mathcal L\cdot0}=\mathrm{id}$ and
$\ee^{\mathcal L(t+s)}=\ee^{\mathcal L t}\ee^{\mathcal L s}$. For the CPTP claim, fix a small step
$\Delta t=t/N_t$ and construct a genuine channel $\Phi$ with
$\norm{\ee^{\mathcal L\Delta t}-\Phi}_\diamond\le C_1\Delta t^2$; we exhibit such a $\Phi$
explicitly in that section (equation the equation above). Granting this, let
$C_2=\sup_{0\le s\le t}\norm{\ee^{\mathcal L s}}_\diamond$. Because the diamond norm is
submultiplicative and $\norm\Phi_\diamond=1$, the telescoping identity
$$\ee^{\mathcal L t}-\Phi^{N_t}
  =\sum_{k=0}^{N_t-1}\ee^{\mathcal L(N_t-1-k)\Delta t}\,
     \bigl(\ee^{\mathcal L\Delta t}-\Phi\bigr)\,\Phi^{k}$$
gives $\norm{\ee^{\mathcal L t}-\Phi^{N_t}}_\diamond\le C_2 C_1 N_t\Delta t^2=C_1C_2\,t\,\Delta t\to0$
as $N_t\to\infty$. Thus $\ee^{\mathcal L t}$ is a diamond-norm limit of the channels
$\Phi^{N_t}$ (a composition of channels is a channel). Since the set of CPTP maps is closed in the
diamond norm, $\ee^{\mathcal L t}$ is itself CPTP.

The converse is the structural GKSL theorem: in finite dimension, the equation above is
the *most general* generator of a norm-continuous semigroup of channels.

**Proposition (GKSL characterization).**

Let $\{\Phi_t\}_{t\ge0}$ be a norm-continuous semigroup of quantum channels on $L(\C^d)$ with
generator $\mathcal G(\rho)=\lim_{t\to0^+}(\Phi_t(\rho)-\rho)/t$. Then there exist a Hermitian
$H$ and operators $L_1,\dots,L_J$ such that $\mathcal G$ has the form of the right-hand side of
the equation above.

*Proof (Proof sketch).*

For small $t$ the channel $\Phi_t$ has a Kraus form $\Phi_t(\rho)=\sum_\alpha K_\alpha(t)\rho
K_\alpha(t)\dg$ with $\sum_\alpha K_\alpha(t)\dg K_\alpha(t)=\id$. Expanding the Choi matrix to
first order in $t$, one may choose the Kraus operators so that one of them is close to the
identity, $K_0(t)=\id+t\mathcal G_0+o(t)$, while the rest are $\mathcal O(\sqrt t)$, namely
$K_j(t)=\sqrt t\,L_j+\mathcal O(t)$. The trace-preservation condition, expanded to first order,
forces $\mathcal G_0+\mathcal G_0\dg+\sum_{j}L_j\dg L_j=0$, so $\mathcal G_0=-\ii H-\tfrac12
\sum_j L_j\dg L_j$ for a Hermitian $H$. Substituting back and collecting the $\mathcal O(t)$
terms of $(\Phi_t(\rho)-\rho)/t$ reproduces the equation above.

##### The adjoint (Heisenberg) equation.

Expectation values of an observable $O$ evolve through the *adjoint* Lindbladian
$\mathcal L\dg$, defined by $\Tr[\mathcal L(\rho)\,O]=\Tr[\rho\,\mathcal L\dg(O)]$ under the
Hilbert--Schmidt pairing $\braket{X|Y}=\Tr[X\dg Y]$:
$$\frac{\dd}{\dd t}\Tr[\rho(t)O]=\Tr[\rho(t)\,\mathcal L\dg(O)],\qquad
  \mathcal L\dg(O)=\ii[H,O]+\sum_{j=1}^{J}\Bigl(L_j\dg O L_j-\tfrac12\{L_j\dg L_j,O\}\Bigr).$$
The Heisenberg picture $O(t)=\ee^{\mathcal L\dg t}(O)$ then satisfies $\Tr[\rho(t)O]=
\Tr[\rho(0)O(t)]$, exactly as for closed systems.

**Example (Classical Markov chains as Lindblad dynamics).**

Take $H=0$ and, for a rate matrix $q_{ij}\ge0$ ($i\ne j$), jump operators
$L_{ij}=\sqrt{q_{ij}}\,\ket{i}\!\bra{j}$. For a diagonal (classical) state
$\rho=\sum_j p_j\proj j$ one has $L_{ij}\rho L_{ij}\dg=q_{ij}p_j\proj i$ and $L_{ij}\dg L_{ij}
=q_{ij}\proj j$, so the equation above keeps $\rho$ diagonal and the populations obey
$\dot p_i=\sum_{j\ne i}q_{ij}p_j-\bigl(\sum_{k\ne i}q_{ki}\bigr)p_i$, i.e. $\dot p=Qp$ with the
classical generator $Q_{ij}=q_{ij}$ ($i\ne j$), $Q_{jj}=-\sum_{i\ne j}q_{ij}$. Restricted to
diagonal states the Lindblad semigroup $\ee^{\mathcal L t}$ is exactly the column-stochastic
Markov semigroup $\ee^{Qt}$: Lindblad dynamics is the quantum analogue of a continuous-time
Markov chain.

**Example (Photon loss in a cavity).**

A single cavity mode with annihilation operator $a$, Hamiltonian $H=\omega a\dg a$, and a single
jump operator $L=\sqrt\kappa\,a$ models photon leakage at rate $\kappa$. With $n=a\dg a$ the
adjoint the equation above gives $\mathcal L\dg(n)=\kappa(a\dg a\dg a a-n^2)=\kappa(n(n-1)
-n^2)=-\kappa n$, so $\dd\langle n\rangle/\dd t=-\kappa\langle n\rangle$ and $\langle n\rangle(t)
=\langle n\rangle(0)\ee^{-\kappa t}$: the mean photon number decays exponentially and the vacuum
$\proj0$ is the steady state.

## Dilation to a larger unitary system

##### The dilated Hamiltonian.

The obstruction to simulating the equation above directly is that $\Phi_t$ is not
unitary. Stinespring's theorem (that theorem) says every channel is a unitary on
a larger space followed by discarding an ancilla; we now realize the *short-time*
Lindblad channel this way, using an ancilla of $a=\lceil\log_2(J+1)\rceil$ qubits with basis
$\{\ket0,\ket1,\dots,\ket J\}$. Define the *dilated Hamiltonian*
$$\widetilde H
  =\proj0\otimes\frac{H}{\sqrt{\Delta t}}
   +\sum_{j=1}^{J}\bigl(\ket j\!\bra0\otimes L_j+\ket0\!\bra j\otimes L_j\dg\bigr)
  =\left(\begin{smallmatrix}
     H/\sqrt{\Delta t} & L_1\dg & \cdots & L_J\dg\\
     L_1 & 0 & \cdots & 0\\
     \vdots & \vdots & & \vdots\\
     L_J & 0 & \cdots & 0
   \end{smallmatrix}\right),$$
and set the short-time unitary $U=\ee^{-\ii\widetilde H\sqrt{\Delta t}}$. The corresponding
channel on the system, obtained by initializing the ancilla in $\ket0$ and tracing it out, is
$$\Phi(\rho)=\Tr_a\!\Bigl(U\bigl(\proj0\otimes\rho\bigr)U\dg\Bigr).$$
The factors of $\sqrt{\Delta t}$ are chosen precisely so that the dissipative terms enter at the
right order. Writing $\widetilde H=\proj0\otimes H/\sqrt{\Delta t}+R$ with $R=\ket j\!\bra0
\otimes L_j+\mathrm{h.c.}$, one has $R^2=\mathrm{diag}(\sum_j L_j\dg L_j,\dots)$, so
$\norm{\widetilde H}^2=\mathcal O(\norm H^2\Delta t+\sum_j\norm{L_j}^2)$ stays bounded once
$\Delta t=\mathcal O(\norm{\mathcal L}_{\mathrm{be}}^{-1})$, where
$$\norm{\mathcal L}_{\mathrm{be}}:=1+\norm H+\sum_{j=1}^{J}\norm{L_j}^2.$$

**Theorem (Dilated-Hamiltonian simulation of Lindblad dynamics).**

Let $\Phi$ be the channel the equation above, $\norm{\mathcal L}_{\mathrm{be}}$ as
in the equation above, and $\Delta t=\mathcal O(\norm{\mathcal L}_{\mathrm{be}}^{-1})$. Then
the one-step error is $\norm{\ee^{\mathcal L\Delta t}-\Phi}_\diamond=\mathcal O
(\norm{\mathcal L}_{\mathrm{be}}^2\,\Delta t^2)$, and hence with $t=N_t\Delta t$,
$$\norm{\ee^{\mathcal L t}-\Phi^{N_t}}_\diamond
  \le N_t\,\norm{\ee^{\mathcal L\Delta t}-\Phi}_\diamond
  =\mathcal O\!\bigl(t\,\norm{\mathcal L}_{\mathrm{be}}^2\,\Delta t\bigr).$$
Consequently, achieving diamond-norm error $\epsilon$ at time $t$ requires
$N_t=\mathcal O(t^2\norm{\mathcal L}_{\mathrm{be}}^2/\epsilon)$ applications of the Hamiltonian
simulation $U=\ee^{-\ii\widetilde H\sqrt{\Delta t}}$, each followed by resetting the ancilla to
$\ket0$.

*Proof (Proof sketch).*

Expand $U=\ee^{-\ii\widetilde H\sqrt{\Delta t}}$ to third order in $\sqrt{\Delta t}$ as $W=\id
-\ii\widetilde H\sqrt{\Delta t}-\tfrac12\widetilde H^2\Delta t+\tfrac{\ii}6\widetilde H^3
\Delta t^{3/2}$, with $\norm{U-W}=\mathcal O(\norm{\widetilde H}^4\Delta t^2)=\mathcal O
(\norm{\mathcal L}_{\mathrm{be}}^2\Delta t^2)$ by the bound $\norm{\widetilde H}^2=\mathcal O
(\norm{\mathcal L}_{\mathrm{be}})$ above. Because $\norm\cdot_\diamond$ is defined over ancillary
extensions of trace-norm-one inputs and the partial trace is a contraction in the trace norm,
replacing $U$ by $W$ in the equation above changes the channel by $\mathcal O
(\norm{\mathcal L}_{\mathrm{be}}^2\Delta t^2)$ in diamond norm. Tracing out the ancilla kills all
odd powers of $\sqrt{\Delta t}$, and a direct expansion of $\Tr_a(W(\proj0\otimes\rho)W\dg)$
yields $\rho+\Delta t\,\mathcal L(\rho)+\mathcal O(\norm{\mathcal L}_{\mathrm{be}}^2\Delta t^2)$,
which matches $\ee^{\mathcal L\Delta t}(\rho)=\rho+\Delta t\,\mathcal L(\rho)+\mathcal O
(\norm{\mathcal L}_{\mathrm{be}}^2\Delta t^2)$. Subtracting gives the one-step bound; the
long-time estimate is the telescoping (linear error growth) argument of that theorem.

This construction, illustrated schematically as a chain of *repeated interactions*---couple
to a fresh ancilla, evolve for $\sqrt{\Delta t}$, discard---turns any Hamiltonian simulation
method of that chapter into a Lindblad simulator.

**Example (Ising evolution with dephasing).**

For the transverse-field Ising model $H=-\sum_{j=1}^{n-1}Z_jZ_{j+1}-g\sum_{j=1}^n X_j$ with
site-local dephasing $L_j=\sqrt\gamma\,Z_j$, the Lindbladian is $\dot\rho=-\ii[H,\rho]+\gamma
\sum_j(Z_j\rho Z_j-\rho)$. The dilated Hamiltonian the equation above uses
$\lceil\log_2(n+1)\rceil$ ancilla qubits with the $L_j=\sqrt\gamma Z_j$ in its first
block-column. With $g,\gamma=\mathcal O(1)$ one has $\norm{\mathcal L}_{\mathrm{be}}=\mathcal O(n)$,
so error $\epsilon$ at time $T$ needs $\Delta t=\mathcal O(\epsilon/(T\norm{\mathcal L}
_{\mathrm{be}}^2))$ and $N_t=\mathcal O(T^2 n^2/\epsilon)$ dilated-Hamiltonian simulation steps.

##### Oblivious amplitude amplification of channels.

Each dilated step succeeds only when the ancilla postselects back to $\ket0$; naively this
success probability compounds and decays, exactly the pathology of that section. The cure
generalizes oblivious amplitude amplification (that chapter) from unitaries to *channels*.
Given a channel $\mathcal Q(\rho)=\sum_{m}K_m\rho K_m\dg$ with $M=2^a$ Kraus operators and a
[select]{.smallcaps} oracle $U_{\textsc{sel}}=\sum_m\proj m\otimes U_{K_m}$, where $U_{K_m}\in
\mathrm{BE}_\alpha(K_m)$, the operator $W=U_{\textsc{sel}}(\Hgate^{\otimes a}\otimes\id)$ is a
block encoding of the Stinespring isometry $V=\sum_m\ket m\otimes K_m$ (that theorem),
with $(\bra{0^b}\otimes\id)W(\ket{0^a}\ket\psi\ket{0^b})=(\alpha\sqrt M)^{-1}V\ket\psi$. Since $V$
is an *isometry*, all its nonzero singular values equal $1$, so amplitude amplification on
the singular value $(\alpha\sqrt M)^{-1}$ is exact: choosing $(\alpha\sqrt M)^{-1}=\sin\frac\pi
2(2k+1)$ maps it to $1$ and eliminates the postselection, yielding a unitary that realizes
$\mathcal Q$ deterministically on the traced-out ancilla. This removes the exponential
success-probability loss from the repeated-interaction chain.

## Dissipative state preparation

##### Fixed points as targets.

A state $\sigma$ is a *fixed point* (stationary state) of the dynamics if $\mathcal L(\sigma)
=0$. For closed systems any $\sigma$ commuting with $H$ is fixed---far too many to be useful. The
dissipative part breaks this degeneracy: a well-chosen $\mathcal L$ can have a *unique* fixed
point $\sigma$, in which case the semigroup is *ergodic* and $\ee^{\mathcal L t}(\rho_0)\to
\sigma$ for every initial $\rho_0$. This is the engine of *dissipative state preparation*:
design $H$ and the $L_j$ so that the target (a thermal state $\ee^{-\beta H_0}/Z$ or a ground
state) is that unique fixed point, then run the dynamics until it mixes. It is the open-system
counterpart of the adiabatic and Zeno solvers of that chapter: instead of slowly deforming a
Hamiltonian whose ground state is tracked, one relaxes irreversibly toward a designed steady
state.

##### Building spectrally selective jump operators.

The construction never diagonalizes $H$. From simple coupling operators $A_a$ (e.g. Paulis) one
forms *filtered* jump operators by convolving the Heisenberg-evolved coupling against a
filter $f_\omega$,
$$L_a(\omega)=\int_{-\infty}^{\infty}f_\omega(s)\,\ee^{\ii Hs}A_a\,\ee^{-\ii Hs}\,\dd s,$$
which, in the eigenbasis $H\ket{\psi_i}=\lambda_i\ket{\psi_i}$, acts as $L_a(\omega)=\sum_{i,k}
\hat f_\omega(\lambda_i-\lambda_k)\braket{\psi_i|A_a|\psi_k}\ket{\psi_i}\!\bra{\psi_k}$, with
$\hat f_\omega$ the Fourier transform of $f_\omega$. Thus the filter $\hat f_\omega$ selects which
energy *transitions* the jump operator drives. Because the equation above is an
integral of Hamiltonian evolutions, a block encoding of $L_a(\omega)$ is assembled by truncating,
discretizing, and applying LCU (that chapter) to the Hamiltonian simulations of
that chapter.

**Proposition (Ground state as a dark state).**

Let $H$ have a nondegenerate ground state $\ket{\psi_0}$ with gap $\Delta=\lambda_1-\lambda_0>0$,
and take a single coupling $A$ and filter with $\hat f(\nu)=1$ for $\nu\le-\Delta$ and
$\hat f(\nu)=0$ for $\nu\ge0$. Then the jump operator $L=\sum_{i,k}\hat f(\lambda_i-\lambda_k)
\braket{\psi_i|A|\psi_k}\ket{\psi_i}\!\bra{\psi_k}$ only lowers energy, so $L\ket{\psi_0}=0$, and
$\proj{\psi_0}$ is a fixed point of the Lindbladian the equation above with $H$ as its
coherent term.

*Proof.*

The support condition on $\hat f$ makes every surviving term $\ket{\psi_i}\!\bra{\psi_k}$ carry
$\lambda_i\le\lambda_k-\Delta<\lambda_k$, so $L$ maps each eigenstate strictly downward in energy
and annihilates the lowest, $L\ket{\psi_0}=0$; hence $L\dg L\ket{\psi_0}=0$ and $L\proj{\psi_0}
L\dg=0$. Since also $[H,\proj{\psi_0}]=0$ ($\ket{\psi_0}$ is an eigenstate), every term of
the equation above annihilates $\proj{\psi_0}$, so $\mathcal L(\proj{\psi_0})=0$. Thus
$\ket{\psi_0}$ is a *dark state*: population flows down the spectrum and pools in the ground
state, which cannot decay further.

Uniqueness of the dark state (and hence convergence) requires that the coupling connect every
excited state to some lower state---an irreducibility condition on $A$---and quantitative
convergence is measured by the *mixing time*
$$\tau_{\mathrm{mix}}(\eta)=\inf\Bigl\{t:\ \sup_{\rho_0}\norm{\ee^{\mathcal L t}(\rho_0)-\sigma}_1
    \le\eta\Bigr\}.$$
On $n$ sites one speaks of *fast mixing* when $\tau_{\mathrm{mix}}=\mathcal O(\poly(n))$ and
*rapid mixing* when $\tau_{\mathrm{mix}}=\mathcal O(\polylog(n))$; proving such bounds (the
open-system analogue of the classical Markov-chain spectral-gap estimates) is problem-specific and
an active research area. The total simulation cost is then $\tau_{\mathrm{mix}}$ multiplied by the
per-time cost of one of the simulators of that section.

##### Unraveling.

For intuition and for classical numerics it helps to represent the same semigroup as an average
over stochastic pure-state trajectories, the *quantum-jump* (Monte Carlo wavefunction)
unraveling: the Itô stochastic Schrödinger equation
$\dd\ket{\psi_t}=\bigl(-\ii H-\tfrac12\sum_j L_j\dg L_j\bigr)\ket{\psi_t}\,\dd t+\sum_j L_j
\ket{\psi_t}\,\dd W_t^j$ satisfies $\E[\proj{\psi_t}]=\rho(t)$ whenever $\E[\proj{\psi_0}]=\rho(0)$.
Between jumps the state evolves under the non-Hermitian effective Hamiltonian $H_{\mathrm{eff}}
=H-\tfrac\ii2\sum_j L_j\dg L_j$; this is exactly the dissipative non-unitary dynamics of
that chapter, and it motivates the truncated Dyson method below.

##### Operator splitting.

An alternative to dilation splits the generator $\mathcal L=\mathcal L_1+\mathcal L_2$ into two
groups (typically coherent versus dissipative) and applies the first-order product
$\Phi(\Delta t)=\ee^{\mathcal L_1\Delta t}\ee^{\mathcal L_2\Delta t}$. The error is governed by
the same superoperator-commutator bound as the Trotter formula (that section):
$$\norm{\ee^{\mathcal L\Delta t}-\ee^{\mathcal L_1\Delta t}\ee^{\mathcal L_2\Delta t}}_\diamond
  \le\tfrac12\Delta t^2\,\norm{[\mathcal L_1,\mathcal L_2]}_\diamond,$$
so if $[\mathcal L_1,\mathcal L_2]=0$ the splitting is exact. With the coarse bounds
$\norm{\mathcal L_H}_\diamond\le2\norm H$ and $\norm{\mathcal D_j}_\diamond\le2\norm{L_j}^2$ for
the single-jump generator $\mathcal D_j(\rho)=L_j\rho L_j\dg-\tfrac12\{L_j\dg L_j,\rho\}$, the
long-time error is $\mathcal O(t\norm{\mathcal L}_{\mathrm{be}}^2\Delta t)$ as in
the equation above. Crucially, unlike the Hamiltonian case, *higher-order*
product formulas cannot be used naïvely: they require negative time steps, but $\ee^{-\mathcal L
t}$ need not be a channel, so one may not run dissipation backwards. Splitting is attractive when
the pieces are exactly solvable: for site-local dephasing $\mathcal D_j$ commutes with
$\mathcal D_k$ and $\ee^{\mathcal D_j\Delta t}(\rho)=\tfrac{1+\ee^{-2\gamma\Delta t}}2\rho+
\tfrac{1-\ee^{-2\gamma\Delta t}}2 Z_j\rho Z_j$ is an elementary single-qubit dephasing channel.

## Nonlinear PDEs via Carleman linearization

##### From nonlinear to infinite-linear.

Quantum evolution is linear, so a nonlinear differential equation must be recast as a linear one
before a quantum solver applies. Consider the quadratic autonomous ODE---the generic form after
semi-discretizing a quadratic PDE such as Navier--Stokes or a reaction--diffusion equation---
$$\frac{\dd z}{\dd t}=F_1 z+F_2\,(z\otimes z),\qquad z(0)=z_0\in\C^d,$$
with $F_1\in\C^{d\times d}$ and $F_2\in\C^{d\times d^2}$. *Carleman linearization* lifts this
to a linear system on the tower of tensor powers $z^{\otimes k}$. Introduce the variables
$y_k:=z^{\otimes k}\in\C^{d^k}$ (so $y_1=z$) and differentiate using the Leibniz rule and
the equation above:
$$\frac{\dd y_k}{\dd t}
  =\sum_{\ell=1}^{k}z^{\otimes(\ell-1)}\otimes\dot z\otimes z^{\otimes(k-\ell)}
  =A_k^k\,y_k+A_k^{k+1}\,y_{k+1},$$
where $A_k^k$ collects the $k$ copies of the *linear* part $F_1$ (each acting on one tensor
slot, the others carried by identities) and $A_k^{k+1}$ collects the $k$ copies of the
*quadratic* part $F_2$, which raises the level from $k+1$ to $k$. Explicitly,
$$A_k^k=\sum_{\ell=1}^{k}\id^{\otimes(\ell-1)}\otimes F_1\otimes\id^{\otimes(k-\ell)},
  \qquad
  A_k^{k+1}=\sum_{\ell=1}^{k}\id^{\otimes(\ell-1)}\otimes F_2\otimes\id^{\otimes(k-\ell)}.$$
Equation the equation above is an *exact* but infinite linear system: the
$k$-th block couples only to itself and the next level $k+1$, an upper-bidiagonal (in block
index) structure. Truncating at level $N$---setting $y_{N+1}\equiv0$---gives a finite linear ODE
$$\frac{\dd}{\dd t}\hat y=\mathcal A\,\hat y,\qquad
  \hat y=(y_1;y_2;\dots;y_N)\in\C^{d+d^2+\dots+d^N},$$
whose block-bidiagonal generator $\mathcal A$ carries $A_k^k$ on the diagonal and $A_k^{k+1}$ on
the super-diagonal. This is precisely a linear ODE of the form treated in that chapter: solve
it with the linear-system, time-marching, LCHS, or Schrödingerization method, then read off
$z(T)$ from the first block $y_1$.

##### Convergence and the dissipativity condition.

The truncation is useful only if the neglected levels contribute negligibly. This is controlled
by a dimensionless ratio comparing the nonlinear forcing to the linear dissipation. Assume $F_1$
is diagonalizable with eigenvalues of negative real part, order them $\mathrm{Re}\,\lambda_d\le
\dots\le\mathrm{Re}\,\lambda_1<0$, and set
$$R:=\frac{\norm{z_0}\,\norm{F_2}}{\abs{\mathrm{Re}\,\lambda_1}}.$$
$R$ is the ratio of the quadratic strength $\norm{F_2}\norm{z_0}$ to the linear decay rate
$\abs{\mathrm{Re}\,\lambda_1}$; it plays the role of a Reynolds number.

**Theorem (Carleman truncation error).**

Suppose $F_1$ is diagonalizable with $\mathrm{Re}\,\lambda_1<0$ and $R<1$ as in
the equation above. Let $z(t)$ solve the equation above and let $\hat y(t)$ solve the
level-$N$ truncation the equation above with the exact initial data
$y_k(0)=z_0^{\otimes k}$. Then the error in the recovered first block obeys
$$\max_{0\le t\le T}\norm{y_1(t)-z(t)}
  \le\norm{z_0}\,R^{N}\,\bigl(1-\ee^{-\abs{\mathrm{Re}\,\lambda_1}t}\bigr)^{N}
  \le\norm{z_0}\,R^{N},$$
so the truncation error decays *geometrically* in the level $N$; achieving accuracy
$\epsilon$ needs only $N=\mathcal O\!\bigl(\log(\norm{z_0}/\epsilon)/\log(1/R)\bigr)$.

*Proof (Proof sketch).*

Diagonalizing $F_1$ and passing to the eigenbasis, the linear part contracts each mode at rate at
least $\abs{\mathrm{Re}\,\lambda_1}$, while the quadratic coupling $A_k^{k+1}$ feeds level $k+1$
into level $k$ with strength $\le k\norm{F_2}$. Writing the error $\eta_k(t)=y_k(t)-z(t)^{\otimes
k}$ (with $\eta_k$ zero above the truncation contributing the sole inhomogeneity at level $N$
through the dropped $A_N^{N+1}y_{N+1}$ term), Duhamel's principle (that result)
bounds $\norm{\eta_k}$ by a convolution of the decaying propagator against the neglected term.
The neglected term at the top level is $\norm{A_N^{N+1}}\norm{y_{N+1}}\le N\norm{F_2}
\norm{z(t)}^{N+1}$, and $\norm{z(t)}\le\norm{z_0}\ee^{-\abs{\mathrm{Re}\,\lambda_1}t}$ under
$R<1$ (the quadratic term cannot overturn the linear decay). Solving the resulting recursion
$\norm{\eta_1}\lesssim\bigl(\norm{F_2}\norm{z_0}/\abs{\mathrm{Re}\,\lambda_1}\bigr)^{N}\norm{z_0}
=R^N\norm{z_0}$, with the extra $(1-\ee^{-\abs{\mathrm{Re}\,\lambda_1}t})^N$ factor from the
finite-time integrals, gives the equation above. The condition $R<1$ is what makes the
geometric series converge; when $R\ge1$ the nonlinear cascade can dominate and the truncation is
not controlled.

**Remark (Cost and caveats).**

The truncated system the equation above has dimension $\sum_{k=1}^N d^k=\mathcal O
(d^N)$, so it uses $\mathcal O(N\log d)$ qubits---exponentially smaller than the classical
$\mathcal O(d^N)$ state, provided the block-encoding oracles for $\mathcal A$ are efficient (they
inherit the sparsity of $F_1,F_2$). Combined with the linear-ODE solvers of that chapter, the
end-to-end cost is $\poly(N,d,\alpha_{\mathcal A}T,\log(1/\epsilon))$ and, because $N=\mathcal O
(\log(1/\epsilon)/\log(1/R))$, polylogarithmic in $1/\epsilon$. The essential restriction is
$R<1$: strongly nonlinear (high-Reynolds) regimes fall outside the guarantee. An alternative for
scalar conservation laws and Hamilton--Jacobi equations is the *level-set* approach, which
lifts the nonlinear equation to a *linear* Liouville (transport) equation in phase space at
the cost of extra dimensions; we do not develop it here.

## Example: Burgers' equation via Cole--Hopf

##### An exact linearization.

Some nonlinear PDEs linearize *exactly*, with no truncation. The archetype is the viscous
Burgers equation, the simplest model combining nonlinear advection with diffusion,
$$\partial_t u+u\,\partial_x u=\nu\,\partial_{xx}u,\qquad \nu>0,$$
where $\nu$ is the viscosity. The *Cole--Hopf transformation* substitutes
$$u=-2\nu\,\frac{\partial_x\phi}{\phi}=-2\nu\,\partial_x(\ln\phi),$$
for a strictly positive potential $\phi(x,t)>0$, and turns the equation above into the
linear heat equation.

**Theorem (Cole--Hopf).**

If $\phi(x,t)>0$ solves the heat equation $\partial_t\phi=\nu\,\partial_{xx}\phi$, then
$u=-2\nu\,\partial_x\phi/\phi$ solves the viscous Burgers equation the equation above.

*Proof.*

Write $u=-2\nu(\ln\phi)_x$. We compute the three terms of the equation above directly. For
the spatial derivatives,
$$u_x=-2\nu(\ln\phi)_{xx}=-2\nu\Bigl(\frac{\phi_{xx}}{\phi}-\frac{\phi_x^2}{\phi^2}\Bigr),
  \qquad
  u_{xx}=-2\nu(\ln\phi)_{xxx}
        =-2\nu\Bigl(\frac{\phi_{xxx}}{\phi}-3\frac{\phi_x\phi_{xx}}{\phi^2}
          +2\frac{\phi_x^3}{\phi^3}\Bigr).$$
The nonlinear term is
$$u\,u_x=\Bigl(-2\nu\frac{\phi_x}{\phi}\Bigr)(-2\nu)
    \Bigl(\frac{\phi_{xx}}{\phi}-\frac{\phi_x^2}{\phi^2}\Bigr)
   =4\nu^2\Bigl(\frac{\phi_x\phi_{xx}}{\phi^2}-\frac{\phi_x^3}{\phi^3}\Bigr).$$
For the time derivative, $u_t=-2\nu(\ln\phi)_{xt}=-2\nu\bigl(\phi_{xt}/\phi-\phi_x\phi_t/\phi^2
\bigr)$. Now use the hypothesis $\phi_t=\nu\phi_{xx}$, hence $\phi_{xt}=\nu\phi_{xxx}$, to get
$$u_t=-2\nu^2\Bigl(\frac{\phi_{xxx}}{\phi}-\frac{\phi_x\phi_{xx}}{\phi^2}\Bigr).$$
Adding the advection term,
$$u_t+u\,u_x
  =-2\nu^2\frac{\phi_{xxx}}{\phi}+2\nu^2\frac{\phi_x\phi_{xx}}{\phi^2}
   +4\nu^2\frac{\phi_x\phi_{xx}}{\phi^2}-4\nu^2\frac{\phi_x^3}{\phi^3}
  =-2\nu^2\frac{\phi_{xxx}}{\phi}+6\nu^2\frac{\phi_x\phi_{xx}}{\phi^2}
   -4\nu^2\frac{\phi_x^3}{\phi^3}.$$
On the other hand,
$$\nu\,u_{xx}=-2\nu^2\Bigl(\frac{\phi_{xxx}}{\phi}-3\frac{\phi_x\phi_{xx}}{\phi^2}
    +2\frac{\phi_x^3}{\phi^3}\Bigr)
  =-2\nu^2\frac{\phi_{xxx}}{\phi}+6\nu^2\frac{\phi_x\phi_{xx}}{\phi^2}
   -4\nu^2\frac{\phi_x^3}{\phi^3}.$$
The two expressions coincide, so $u_t+u u_x=\nu u_{xx}$, as claimed.

**Remark (Why the substitution is guessed).**

The choice the equation above is natural in two steps. First set $u=\partial_x\psi$
(assuming $u$ has a potential); Burgers becomes $\partial_x(\psi_t+\tfrac12\psi_x^2-\nu\psi_{xx})
=0$, so up to an $x$-independent function $\psi_t+\tfrac12\psi_x^2=\nu\psi_{xx}$, a
Hamilton--Jacobi equation. Second, the exponential (Hopf--Cole) substitution $\psi=-2\nu\ln\phi$
converts the quadratic $\tfrac12\psi_x^2$ into a linear term: $\psi_t=-2\nu\phi_t/\phi$,
$\psi_x=-2\nu\phi_x/\phi$, $\psi_{xx}=-2\nu(\phi_{xx}/\phi-\phi_x^2/\phi^2)$, and substituting
collapses everything to $\phi_t=\nu\phi_{xx}$. The nonlinearity is absorbed because the same
$\phi_x^2/\phi^2$ term appears in $\tfrac12\psi_x^2$ and in $\nu\psi_{xx}$ with matching
coefficients.

##### The quantum algorithm.

Cole--Hopf reduces solving Burgers to three classical/quantum steps. (i) Transform the initial
data: given $u(x,0)=u_0(x)$, set $\phi(x,0)=\exp\bigl(-\tfrac1{2\nu}\int_0^x u_0(x')\,\dd x'\bigr)
>0$, which is exactly the inversion of the equation above at $t=0$. (ii) Solve the linear
heat equation $\partial_t\phi=\nu\partial_{xx}\phi$ to time $T$ by any method of that chapter
(linear-system, LCHS, or Schrödingerization); after spatial discretization this is the
tridiagonal Laplacian system of that section with $\alpha_A=\mathcal O(\nu N^2)$. (iii)
Recover $u(x,T)=-2\nu\,\partial_x\phi/\phi$, a division and a derivative on the grid. Because
$\phi>0$ stays strictly positive (the heat semigroup preserves positivity, and a positive initial
$\phi$ stays positive), the recovery is numerically well posed. The nonlinear solve thus inherits
the near-optimal $\widetilde\bigo(q\,\nu N^2 T)$ scaling of the linear heat-equation solver, with
the norm-growth factor $q$ now referring to the potential $\phi$.

**Remark (Scope).**

Cole--Hopf is special to Burgers (and a handful of related equations); general quadratic PDEs
have no exact linearization and require the truncated Carleman route of that section, whose
guarantee is conditional on $R<1$. Together the two examples bracket the state of the art:
*exact* linearization where a lucky change of variables exists, and *approximate*
linearization with controlled error in the weakly nonlinear regime otherwise.

## Exercises {#exercises .unnumbered}

**Exercise.**

*Trace preservation and the adjoint.* (a) Show directly from the equation above that
$\Tr[\mathcal L(\rho)]=0$ for every $\rho$, so the Lindblad flow preserves the trace. (b) Verify
that $\mathcal L\dg$ in the equation above is the Hilbert--Schmidt adjoint of $\mathcal L$,
i.e. $\Tr[\mathcal L(X)\dg Y]=\Tr[X\dg\mathcal L\dg(Y)]$ for all $X,Y$, and confirm that
$\mathcal L\dg(\id)=0$ (unitality of the Heisenberg dynamics).

**Exercise.**

*Classical Markov chains.* Complete the computation of that example: with $H=0$
and $L_{ij}=\sqrt{q_{ij}}\,\ket i\!\bra j$, substitute into the equation above and show
that a diagonal $\rho=\sum_j p_j\proj j$ stays diagonal with $\dot p=Qp$, identifying the entries
of $Q$. Verify that each column of $Q$ sums to zero, so $\ee^{Qt}$ is column-stochastic.

**Exercise.**

*The dilated Hamiltonian reproduces $\mathcal L$.* For a single jump operator $L_1$, take
$\widetilde H$ from the equation above and the channel the equation above.
Expand $U=\ee^{-\ii\widetilde H\sqrt{\Delta t}}$ to second order, trace out the ancilla, and show
$\Tr_a(U(\proj0\otimes\rho)U\dg)=\rho+\Delta t\,\mathcal L(\rho)+\mathcal O(\Delta t^2)$. Explain
why the odd powers of $\sqrt{\Delta t}$ vanish under the partial trace, and why the factor
$H/\sqrt{\Delta t}$ in the equation above is needed for the coherent term to appear at order
$\Delta t$.

**Exercise.**

*Photon loss.* For the cavity model of that example with $H=\omega a\dg a$ and
$L=\sqrt\kappa\,a$, use the adjoint Lindbladian to compute $\mathcal L\dg(n)$ for $n=a\dg a$ and
deduce $\langle n\rangle(t)=\langle n\rangle(0)\ee^{-\kappa t}$. Then compute $\mathcal L\dg(a)$
and find the decay of the coherence $\langle a\rangle(t)$.

**Exercise.**

*Dark state and fixed point.* Prove that result: given the energy-lowering
filter, show $L\ket{\psi_0}=0$ and that every term of the Lindbladian annihilates
$\proj{\psi_0}$, so $\mathcal L(\proj{\psi_0})=0$. Explain what additional structural condition on
the coupling $A$ is needed for $\proj{\psi_0}$ to be the *unique* fixed point, and relate the
convergence rate to the mixing time the equation above.

**Exercise.**

*Carleman linearization.* Starting from the quadratic ODE the equation above, derive
the block recursion the equation above--the equation above for
$y_k=z^{\otimes k}$ by differentiating $z^{\otimes k}$ with the product rule and substituting
$\dot z=F_1 z+F_2(z\otimes z)$. Explain why the resulting infinite system is block upper-bidiagonal
in the level index $k$, and why truncating at level $N$ (setting $y_{N+1}=0$) leaves a finite
linear ODE solvable by the methods of that chapter. State the role of the ratio $R$ of
the equation above in controlling the truncation error.

**Exercise.**

*Cole--Hopf details.* (a) Re-derive that theorem through the potential
substitution of that remark: set $u=\psi_x$, obtain the Hamilton--Jacobi
equation $\psi_t+\tfrac12\psi_x^2=\nu\psi_{xx}$, then substitute $\psi=-2\nu\ln\phi$ and show it
collapses to $\phi_t=\nu\phi_{xx}$. (b) Verify that the initial-data transform $\phi(x,0)=
\exp\bigl(-\tfrac1{2\nu}\int_0^x u_0\bigr)$ inverts the equation above at $t=0$, and argue
that $\phi>0$ is preserved by the heat flow so the recovery $u=-2\nu\phi_x/\phi$ is well defined
for all $t$.

## Further reading {#further-reading .unnumbered}

-   Lin Lin, *live notes*, Ch. 21 (open quantum systems): the Lindblad equation and the
    proof that $\ee^{\mathcal L t}$ is CPTP, the GKSL characterization, dissipative thermal
    and ground-state preparation, the dilated-Hamiltonian (Stinespring) simulation, oblivious
    amplitude amplification of channels, the operator-splitting method, and the truncated
    Dyson method.

-   H.-P. Breuer and F. Petruccione, *The Theory of Open Quantum Systems*, Oxford
    University Press, 2002, for physical background on Lindblad dynamics, quantum-jump
    unravelings, and Markovian master equations.

-   G. Lindblad, *On the generators of quantum dynamical semigroups*, Commun. Math. Phys. **48**, 119 (1976); and V. Gorini, A. Kossakowski, and E. C. G. Sudarshan,
    *Completely positive dynamical semigroups of $N$-level systems*, J. Math. Phys. **17**, 821 (1976), for the original GKSL theorem of that result.

-   J.-P. Liu, H. Ø. Kolden, H. K. Krovi, N. F. Loureiro, K. Trivisa, and A. M. Childs,
    *Efficient quantum algorithm for dissipative nonlinear differential equations*, Proc. Natl. Acad. Sci. USA **118**, e2026805118 (2021) (arXiv:2011.03185), for the
    Carleman-linearization algorithm and the $R<1$ convergence condition of
    that section.

-   J. D. Cole, *On a quasi-linear parabolic equation occurring in aerodynamics*, Quart. Appl. Math. **9**, 225 (1951); and E. Hopf, *The partial differential equation
    $u_t+uu_x=\mu u_{xx}$*, Comm. Pure Appl. Math. **3**, 201 (1950), for the exact
    linearization of Burgers' equation (that section); see also L. C. Evans,
    *Partial Differential Equations*, 2nd ed., AMS, 2010, §4.4.

-   S. Jin and N. Liu, *Quantum algorithms for nonlinear partial differential equations*,
    Bull. Sci. Math. **194**, 103457 (2024) (arXiv:2212.00681), for the level-set /
    Liouville (Hamilton--Jacobi) approach mentioned in that remark.